In [0]:
# ============================================================================
# Proyecto Medallion - Catalogo "cobranzas"
# Paso 3: Silver (cuentas) -> Gold (star schema, cobranzas.gold.*)
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

CATALOG = "cobranzas"
GOLD = f"{CATALOG}.gold"
SILVER_CUENTAS = f"{CATALOG}.silver.cuentas"

# Dimensiones
DIM_RESPONSABLES = f"{GOLD}.dim_responsables_cuenta"
DIM_CLIENTES = f"{GOLD}.dim_clientes"
DIM_CONTACTO_CLIENTES = f"{GOLD}.dim_contacto_clientes"
DIM_SEGMENTACION = f"{GOLD}.dim_segmentacion_productos"
DIM_TIPOS_PRODUCTO = f"{GOLD}.dim_tipos_producto"
DIM_PRODUCTOS = f"{GOLD}.dim_productos"
DIM_ESTADOS_CARTERA = f"{GOLD}.dim_estados_cartera"
DIM_TIPOS_EMPRESAS = f"{GOLD}.dim_tipos_empresas"
DIM_EMPRESAS = f"{GOLD}.dim_empresas"
DIM_BIENES = f"{GOLD}.dim_bienes_inmuebles"
DIM_CARACT_MENSUAL = f"{GOLD}.dim_caracterizaciones_mensuales"
DIM_SUB_CARACT = f"{GOLD}.dim_sub_caracterizaciones"
DIM_CODIGOS_GESTION = f"{GOLD}.dim_codigos_gestion"
DIM_ESTADOS_PROMESA = f"{GOLD}.dim_estados_promesa"
DIM_TIPOS_DEMANDA = f"{GOLD}.dim_tipos_demanda"
DIM_ESTADOS_DEMANDA = f"{GOLD}.dim_estados_demanda"
DIM_CARACT_JUDICIAL = f"{GOLD}.dim_caracterizaciones_judiciales"
DIM_EMBARGOS_DE = f"{GOLD}.dim_embargos_de"
DIM_JUZGADOS = f"{GOLD}.dim_juzgados"
DIM_JUECES = f"{GOLD}.dim_jueces"
DIM_CIUDADES = f"{GOLD}.dim_ciudades"
BRIDGE_CIUDADES = f"{GOLD}.bridge_contacto_ciudades"
DIM_TIPO_ARREGLO = f"{GOLD}.dim_tipo_arreglo"

# Hechos
FACT_CUENTA = f"{GOLD}.fact_cuenta"
FACT_PROMESAS = f"{GOLD}.fact_promesas"
FACT_LABORAL = f"{GOLD}.fact_laboral_clientes"
FACT_GESTION_CUENTA = f"{GOLD}.fact_gestion_cuenta"
FACT_GESTION_DETALLE = f"{GOLD}.fact_gestion_detalle"
FACT_EMBARGOS = f"{GOLD}.fact_embargos"
FACT_GESTION_JUDICIAL_CUENTA = f"{GOLD}.fact_gestion_judicial_cuenta"
FACT_GESTION_JUDICIAL_DETALLE = f"{GOLD}.fact_gestion_judicial_detalle"

# COMMAND ----------

# ---------------------------------------------------------------------------
# 0. Motor generico: upsert de dimensiones, SCD2 y resolucion de FKs
# ---------------------------------------------------------------------------

def _short(table: str) -> str:
    return table.split(".")[-1]


def upsert_catalog(table: str, key_cols: list, source_df: DataFrame, extra_insert_cols: dict = None):
    """
    Inserta combinaciones NUEVAS de key_cols que no existan aun en la tabla.
    No actualiza filas existentes (para eso esta merge_dimension_with_attrs).
    extra_insert_cols: dict columna -> expresion SQL literal fija para el
    INSERT (p.ej. {"actual": "true", "fecha_inicio": "current_timestamp()"}).
    """
    view = f"stg_{_short(table)}"
    not_null_filter = " AND ".join([f"{c} IS NOT NULL" for c in key_cols])
    (
        source_df.filter(not_null_filter)
        .select(*key_cols)
        .dropDuplicates(key_cols)
        .createOrReplaceTempView(view)
    )
    join_cond = " AND ".join([f"t.{c} <=> s.{c}" for c in key_cols])
    extra_cols = list(extra_insert_cols.keys()) if extra_insert_cols else []
    insert_cols = ", ".join(key_cols + extra_cols)
    select_vals = ", ".join([f"s.{c}" for c in key_cols] + [extra_insert_cols[c] for c in extra_cols])
    spark.sql(f"""
        INSERT INTO {table} ({insert_cols})
        SELECT {select_vals}
        FROM {view} s
        LEFT ANTI JOIN {table} t ON {join_cond}
    """)


def merge_dimension_with_attrs(table: str, key_col: str, attr_cols: list, source_df: DataFrame):
    """
    MERGE clasico: actualiza atributos si la llave ya existe, inserta si no.
    Sirve tanto para dimensiones con FK (dim_productos, dim_empresas,
    dim_clientes) como para hechos de "estado actual" (fact_cuenta,
    fact_gestion_cuenta, etc.) cuya llave de negocio es una sola columna.
    """
    view = f"stg_{_short(table)}"
    source_df.filter(f"{key_col} IS NOT NULL").dropDuplicates([key_col]).createOrReplaceTempView(view)
    set_clause = ", ".join([f"t.{c} = s.{c}" for c in attr_cols] + ["t._updated_at = current_timestamp()"])
    insert_cols = ", ".join([key_col] + attr_cols)
    insert_vals = ", ".join([f"s.{c}" for c in [key_col] + attr_cols])
    spark.sql(f"""
        MERGE INTO {table} t
        USING {view} s
        ON t.{key_col} = s.{key_col}
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
    """)


def scd2_upsert(table: str, natural_key_col: str, attr_cols: list, source_df: DataFrame):
    """
    SCD2 real: si natural_key_col es nuevo, inserta version actual=true.
    Si ya existe y algun attr_col cambio respecto a la version actual=true,
    cierra la version vieja (actual=false, fecha_fin=ahora) e inserta una
    version nueva (actual=true, fecha_inicio=ahora).
    """
    short = _short(table)
    view = f"stg_{short}"
    current_view = f"cur_{short}"
    changed_view = f"chg_{short}"
    ins_view = f"ins_{short}"

    source_df.filter(f"{natural_key_col} IS NOT NULL").dropDuplicates([natural_key_col]).createOrReplaceTempView(view)
    spark.table(table).filter("actual = true").createOrReplaceTempView(current_view)

    diff_condition = " OR ".join([f"NOT (c.{c} <=> s.{c})" for c in attr_cols])
    spark.sql(f"""
        SELECT s.{natural_key_col}
        FROM {view} s
        JOIN {current_view} c ON c.{natural_key_col} = s.{natural_key_col}
        WHERE {diff_condition}
    """).createOrReplaceTempView(changed_view)

    spark.sql(f"""
        UPDATE {table} t
        SET actual = false, fecha_fin = current_timestamp()
        WHERE actual = true
        AND {natural_key_col} IN (SELECT {natural_key_col} FROM {changed_view})
    """)

    attr_select = ", ".join([f"s.{c}" for c in attr_cols])
    spark.sql(f"""
        SELECT s.{natural_key_col}, {attr_select}
        FROM {view} s
        LEFT ANTI JOIN {current_view} c ON c.{natural_key_col} = s.{natural_key_col}
        UNION ALL
        SELECT s.{natural_key_col}, {attr_select}
        FROM {view} s
        JOIN {changed_view} ch ON ch.{natural_key_col} = s.{natural_key_col}
    """).createOrReplaceTempView(ins_view)

    insert_cols = ", ".join([natural_key_col] + attr_cols + ["actual", "fecha_inicio", "fecha_fin"])
    spark.sql(f"""
        INSERT INTO {table} ({insert_cols})
        SELECT *, true, current_timestamp(), CAST(NULL AS TIMESTAMP)
        FROM {ins_view}
    """)


def attach_id(df: DataFrame, table: str, key_cols: list, id_col: str, source_cols: list, target_alias: str) -> DataFrame:
    """
    Left join de df (por source_cols) contra table (por key_cols) para traer
    id_col con el nombre target_alias. Si la tabla destino tiene columna
    'actual' (dimensiones SCD2), filtra automaticamente a la version vigente.
    """
    dim = spark.table(table)
    if "actual" in dim.columns:
        dim = dim.filter("actual = true")
    lk_aliases = [f"_lk_{i}" for i in range(len(key_cols))]
    dim = dim.select(*[F.col(k).alias(a) for k, a in zip(key_cols, lk_aliases)], F.col(id_col).alias(target_alias))
    cond = None
    for a, s in zip(lk_aliases, source_cols):
        c = df[s].eqNullSafe(dim[a])
        cond = c if cond is None else (cond & c)
    joined = df.join(dim, cond, "left")
    return joined.drop(*lk_aliases)

# ---------------------------------------------------------------------------
# 1. Leer silver.cuentas
# ---------------------------------------------------------------------------
silver = spark.table(SILVER_CUENTAS)

# ---------------------------------------------------------------------------
# 2. Dimensiones simples (catalogo de 1 columna, insert-only)
# ---------------------------------------------------------------------------
SIMPLE_DIMS = [
    ("estado_cartera", DIM_ESTADOS_CARTERA, "estado"),
    ("segmentacion_producto", DIM_SEGMENTACION, "segmento"),
    ("tipo_producto", DIM_TIPOS_PRODUCTO, "tipo"),
    ("tipo_empresa", DIM_TIPOS_EMPRESAS, "tipo_empresa"),
    ("bienes_inmuebles", DIM_BIENES, "bien_inmueble"),
    ("caracterizacion_del_mes", DIM_CARACT_MENSUAL, "caracterizacion_mensual"),
    ("sub_caracterizacion", DIM_SUB_CARACT, "sub_caracterizacion"),
    ("codigos_gestion", DIM_CODIGOS_GESTION, "codigos_gestion"),
    ("estado_promesa", DIM_ESTADOS_PROMESA, "estado_promesa"),
    ("tipo_demanda", DIM_TIPOS_DEMANDA, "tipo_demanda"),
    ("estado_demanda", DIM_ESTADOS_DEMANDA, "estado_demanda"),
    ("caracterizacion_judicial", DIM_CARACT_JUDICIAL, "caracterizacion_judicial"),
    ("embargo_de", DIM_EMBARGOS_DE, "embargo_de"),
    ("juzgado", DIM_JUZGADOS, "juzgado"),
    ("juez", DIM_JUECES, "juez"),
    ("tipo_arreglo_promesa", DIM_TIPO_ARREGLO, "tipo_arreglo"),
]

for silver_col, table, business_col in SIMPLE_DIMS:
    values = silver.select(F.col(silver_col).alias(business_col))
    upsert_catalog(table, [business_col], values)

# ---------------------------------------------------------------------------
# 3. Dimensiones con atributos/FK: dim_productos, dim_empresas
# ---------------------------------------------------------------------------

# 3.1 dim_productos (producto -> segmento_id, tipo_id)
productos_src = (
    silver.select("producto", "segmentacion_producto", "tipo_producto")
    .filter(F.col("producto").isNotNull())
    .dropDuplicates(["producto", "segmentacion_producto", "tipo_producto"])
)
productos_src = attach_id(productos_src, DIM_SEGMENTACION, ["segmento"], "segmento_id", ["segmentacion_producto"], "segmento_id")
productos_src = attach_id(productos_src, DIM_TIPOS_PRODUCTO, ["tipo"], "tipo_id", ["tipo_producto"], "tipo_id")
# si un mismo producto aparece con mas de una combinacion segmento/tipo (dato
# sucio), nos quedamos con la que mas atributos resuelve
w = Window.partitionBy("producto").orderBy(F.col("segmento_id").isNull().asc(), F.col("tipo_id").isNull().asc())
productos_src = productos_src.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
merge_dimension_with_attrs(DIM_PRODUCTOS, "producto", ["segmento_id", "tipo_id"], productos_src)

# 3.2 dim_empresas (empresa = lugar_trabajo -> tipo_empresa_id)
empresas_src = (
    silver.select(F.col("lugar_trabajo").alias("empresa"), "tipo_empresa", "silver_processed_at")
    .filter(F.col("empresa").isNotNull())
)
w = Window.partitionBy("empresa").orderBy(F.col("silver_processed_at").desc())
empresas_src = empresas_src.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
empresas_src = attach_id(empresas_src, DIM_TIPOS_EMPRESAS, ["tipo_empresa"], "tipo_empresa_id", ["tipo_empresa"], "tipo_empresa_id")
merge_dimension_with_attrs(DIM_EMPRESAS, "empresa", ["tipo_empresa_id"], empresas_src)

# ---------------------------------------------------------------------------
# 4. dim_clientes, dim_contacto_clientes (SCD2), dim_responsables_cuenta
# ---------------------------------------------------------------------------

# 4.1 dim_clientes: 1 fila por numero_cliente_unico, atributos de la carga mas reciente
clientes_src = (
    silver.select("numero_cliente_unico", "nombre", "dni", "bienes_inmuebles", "silver_processed_at")
    .filter(F.col("numero_cliente_unico").isNotNull())
)
w = Window.partitionBy("numero_cliente_unico").orderBy(F.col("silver_processed_at").desc())
clientes_src = clientes_src.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
clientes_src = attach_id(clientes_src, DIM_BIENES, ["bien_inmueble"], "bien_inmueble_id", ["bienes_inmuebles"], "bien_inmueble_id")
merge_dimension_with_attrs(DIM_CLIENTES, "numero_cliente_unico", ["nombre", "dni", "bien_inmueble_id"], clientes_src)

# 4.2 dim_contacto_clientes: SCD2 real por cliente_id (email/telefonos/celular)
contacto_src = (
    silver.select("numero_cliente_unico", "email", "telefonos", "celular", "silver_processed_at")
    .filter(F.col("numero_cliente_unico").isNotNull())
)
w = Window.partitionBy("numero_cliente_unico").orderBy(F.col("silver_processed_at").desc())
contacto_src = contacto_src.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
contacto_src = attach_id(contacto_src, DIM_CLIENTES, ["numero_cliente_unico"], "cliente_id", ["numero_cliente_unico"], "cliente_id")
contacto_src = contacto_src.filter(F.col("cliente_id").isNotNull())
scd2_upsert(DIM_CONTACTO_CLIENTES, "cliente_id", ["email", "telefonos", "celular"], contacto_src)

# 4.3 dim_responsables_cuenta: catalogo insert-only de combinaciones (jefatura, gestor)

responsables_src = silver.select("jefatura", "gestor")
upsert_catalog(
    DIM_RESPONSABLES,
    ["jefatura", "gestor"],
    responsables_src,
    extra_insert_cols={"actual": "true", "fecha_inicio": "current_timestamp()"},
)

# ---------------------------------------------------------------------------
# 5. dim_ciudades + bridge_contacto_ciudades
# ---------------------------------------------------------------------------

# Mismo diccionario de normalizacion que "ciudad" en COLUMN_CONFIG de silver
# (03_bronze_to_silver_cuentas.py) 
CIUDAD_NORMALIZE = {
    "San Pedro Sula": [
        "SAN PEDRO SULA", "san pedro sula", "San Pedro Sua",
        "Sa Pedro Sula", "San Pedro Sula El Progreso",
    ],
    "Choloma": ["CHOLOMA", "choloma", "Choloms"],
    "La Ceiba": ["LA CEIBA", "la ceiba"],
    "Puerto Cortés": ["PUERTO CORTÉS", "Puerto Cortes", "puerto cortes"],
    "Villanueva": ["VILLANUEVA", "villanueva"],
    "El Progreso": ["EL PROGRESO", "el progreso"],
    "Tela": ["TELA", "tela"],
    "Santa Cruz de Yojoa": ["Santa Cruz De Yojoa", "SANTA CRUZ DE YOJOA"],
    "Sin Informacion": ["101", "???"],
    "Cofradía": ["Cofradia"],
    "Tegucigalpa": ["TEGUCIGALPA", "tegucigalpa"],
}


def normalize_lookup(col: "F.Column", normalize_map: dict) -> "F.Column":
    flat = {}
    for canonical, variants in normalize_map.items():
        flat[canonical] = canonical
        for v in variants:
            flat[v] = canonical
    pairs = []
    for k, v in flat.items():
        pairs.extend([F.lit(k), F.lit(v)])
    lookup = F.create_map(*pairs)
    return F.coalesce(lookup[col], col)


# 5.1 ciudad (silver.cuentas.ciudad): SI puede traer varias ciudades separadas
# por coma/slash -> split + explode + normalizar
ciudad_split = (
    silver.select("numero_de_prestamo_vp", "numero_cliente_unico", "ciudad")
    .filter(F.col("ciudad").isNotNull())
    .withColumn("ciudad_parte", F.explode(F.split(F.col("ciudad"), r"[,/]+")))
    .withColumn("ciudad_parte", F.initcap(F.trim(F.col("ciudad_parte"))))
    .filter((F.col("ciudad_parte").isNotNull()) & (F.col("ciudad_parte") != ""))
    .withColumn("ciudad_norm", normalize_lookup(F.col("ciudad_parte"), CIUDAD_NORMALIZE))
)

# 5.2 poblar dim_ciudades con las ciudades de "ciudad" (split) + "ciudad_demanda" (ya normalizada en silver, un solo valor)
ciudades_a_insertar = ciudad_split.select(F.col("ciudad_norm").alias("ciudad")).unionByName(
    silver.select(F.col("ciudad_demanda").alias("ciudad")).filter(F.col("ciudad_demanda").isNotNull())
)
upsert_catalog(DIM_CIUDADES, ["ciudad"], ciudades_a_insertar)

# 5.3 bridge_contacto_ciudades: ciudades distintas entre todas las cuentas de un mismo cliente
ciudad_por_cliente = (
    ciudad_split.select("numero_cliente_unico", "ciudad_norm")
    .filter(F.col("numero_cliente_unico").isNotNull())
    .dropDuplicates()
)
ciudad_por_cliente = attach_id(ciudad_por_cliente, DIM_CLIENTES, ["numero_cliente_unico"], "cliente_id", ["numero_cliente_unico"], "cliente_id")
ciudad_por_cliente = attach_id(ciudad_por_cliente, DIM_CIUDADES, ["ciudad"], "ciudad_id", ["ciudad_norm"], "ciudad_id")
ciudad_por_cliente = attach_id(ciudad_por_cliente, DIM_CONTACTO_CLIENTES, ["cliente_id"], "contacto_cliente_id", ["cliente_id"], "contacto_cliente_id")

bridge_src = (
    ciudad_por_cliente
    .filter(F.col("contacto_cliente_id").isNotNull() & F.col("ciudad_id").isNotNull())
    .select("contacto_cliente_id", "ciudad_id")
    .dropDuplicates()
)
upsert_catalog(BRIDGE_CIUDADES, ["contacto_cliente_id", "ciudad_id"], bridge_src)

# ---------------------------------------------------------------------------
# 6. fact_cuenta (grano: 1 fila por numero_de_prestamo_vp, estado actual)
# ---------------------------------------------------------------------------
cuenta_base = (
    silver.withColumn(
        "_rn",
        F.row_number().over(Window.partitionBy("numero_de_prestamo_vp").orderBy(F.col("silver_processed_at").desc())),
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

cuenta_stg = cuenta_base.select(
    "numero_de_prestamo_vp", "numero_cliente_unico", "jefatura", "gestor",
    "producto", "estado_cartera", "fecha_asignacion", "fecha_castigo",
    "saldo_l", "saldo_usd", "saldo_total_usd", "saldo_total_honorarios", "fecha_ultimo_pago",
)
cuenta_stg = attach_id(cuenta_stg, DIM_CLIENTES, ["numero_cliente_unico"], "cliente_id", ["numero_cliente_unico"], "cliente_id")
cuenta_stg = attach_id(cuenta_stg, DIM_RESPONSABLES, ["jefatura", "gestor"], "responsables_cuenta_id", ["jefatura", "gestor"], "responsables_cuenta_id")
cuenta_stg = attach_id(cuenta_stg, DIM_PRODUCTOS, ["producto"], "producto_id", ["producto"], "producto_id")
cuenta_stg = attach_id(cuenta_stg, DIM_ESTADOS_CARTERA, ["estado"], "estado_id", ["estado_cartera"], "estado_cartera_id")

FACT_CUENTA_ATTR_COLS = [
    "cliente_id", "responsables_cuenta_id", "producto_id", "estado_cartera_id",
    "fecha_asignacion", "fecha_castigo", "saldo_l", "saldo_usd",
    "saldo_total_usd", "saldo_total_honorarios", "fecha_ultimo_pago",
]
merge_dimension_with_attrs(FACT_CUENTA, "numero_de_prestamo_vp", FACT_CUENTA_ATTR_COLS, cuenta_stg)

# ---------------------------------------------------------------------------
# 7. fact_promesas, fact_laboral_clientes, fact_embargos
# ---------------------------------------------------------------------------

# 7.1 fact_promesas: solo filas de silver que SI traen promesa
promesas_stg = (
    silver.filter(F.col("fecha_de_pago_promesa").isNotNull() | F.col("estado_promesa").isNotNull())
    .select(
        "numero_de_prestamo_vp", "tipo_arreglo_promesa", "monto_promesa", "fecha_promesa",
        "monto_mensual_lps_promesa", "monto_mensual_dolar_promesa", "fecha_de_pago_promesa",
        "meses_promesa", "estado_promesa", "fecha_inicio_promesa", "fecha_final_promesa",
    )
    .dropDuplicates()
)
promesas_stg = attach_id(promesas_stg, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
promesas_stg = attach_id(promesas_stg, DIM_TIPO_ARREGLO, ["tipo_arreglo"], "tipo_arreglo_id", ["tipo_arreglo_promesa"], "tipo_arreglo_id")
promesas_stg = attach_id(promesas_stg, DIM_ESTADOS_PROMESA, ["estado_promesa"], "estado_promesa_id", ["estado_promesa"], "estado_promesa_id")
promesas_stg = (
    promesas_stg
    .withColumnRenamed("monto_mensual_lps_promesa", "monto_mensual_l")
    .withColumnRenamed("monto_mensual_dolar_promesa", "monto_mensual_usd")
    .withColumnRenamed("fecha_de_pago_promesa", "fecha_de_pago")
    .withColumnRenamed("meses_promesa", "meses")
    .withColumnRenamed("fecha_inicio_promesa", "fecha_inicio")
    .withColumnRenamed("fecha_final_promesa", "fecha_fin")
    .withColumn("cumplida", F.col("estado_promesa") == F.lit("Cumplida"))  # ver supuesto D
    .filter(F.col("cuenta_id").isNotNull())
)

promesas_stg.createOrReplaceTempView("stg_fact_promesas")
spark.sql(f"""
    MERGE INTO {FACT_PROMESAS} t
    USING stg_fact_promesas s
    ON t.cuenta_id = s.cuenta_id
       AND t.fecha_de_pago <=> s.fecha_de_pago
       AND t.estado_promesa_id <=> s.estado_promesa_id
    WHEN MATCHED THEN UPDATE SET
        t.tipo_arreglo_id = s.tipo_arreglo_id,
        t.monto_promesa = s.monto_promesa,
        t.fecha_promesa = s.fecha_promesa,
        t.monto_mensual_l = s.monto_mensual_l,
        t.monto_mensual_usd = s.monto_mensual_usd,
        t.meses = s.meses,
        t.cumplida = s.cumplida,
        t.fecha_inicio = s.fecha_inicio,
        t.fecha_fin = s.fecha_fin,
        t._updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
        cuenta_id, tipo_arreglo_id, monto_promesa, fecha_promesa, monto_mensual_l,
        monto_mensual_usd, fecha_de_pago, meses, estado_promesa_id, cumplida,
        fecha_inicio, fecha_fin
    ) VALUES (
        s.cuenta_id, s.tipo_arreglo_id, s.monto_promesa, s.fecha_promesa, s.monto_mensual_l,
        s.monto_mensual_usd, s.fecha_de_pago, s.meses, s.estado_promesa_id, s.cumplida,
        s.fecha_inicio, s.fecha_fin
    )
""")

# 7.2 fact_laboral_clientes: 1 fila por cliente, de la carga mas reciente
laboral_stg = (
    cuenta_base.select("numero_cliente_unico", "base_laboral", "lugar_trabajo", "silver_processed_at")
    .filter(F.col("numero_cliente_unico").isNotNull())
)
w = Window.partitionBy("numero_cliente_unico").orderBy(F.col("silver_processed_at").desc())
laboral_stg = laboral_stg.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
laboral_stg = attach_id(laboral_stg, DIM_CLIENTES, ["numero_cliente_unico"], "cliente_id", ["numero_cliente_unico"], "cliente_id")
laboral_stg = attach_id(laboral_stg, DIM_EMPRESAS, ["empresa"], "empresa_id", ["lugar_trabajo"], "empresa_id")
laboral_stg = (
    laboral_stg.withColumn("labora", F.col("base_laboral") == F.lit("Si"))
    .filter(F.col("cliente_id").isNotNull())
)
merge_dimension_with_attrs(FACT_LABORAL, "cliente_id", ["labora", "empresa_id"], laboral_stg)

# 7.3 fact_embargos: 1 fila por cuenta, solo si hay datos de embargo
embargos_stg = (
    cuenta_base
    .filter(F.col("fecha_embargo").isNotNull() | F.col("embargo_de").isNotNull())
    .select("numero_de_prestamo_vp", "embargo_de", "fecha_embargo", "cantidad_mensual_retenida", "lugar_trabajo")
)
embargos_stg = attach_id(embargos_stg, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
embargos_stg = attach_id(embargos_stg, DIM_EMBARGOS_DE, ["embargo_de"], "embargo_de_id", ["embargo_de"], "embargo_de_id")
embargos_stg = attach_id(embargos_stg, DIM_EMPRESAS, ["empresa"], "empresa_id", ["lugar_trabajo"], "empresa_id_raw")
embargos_stg = (
    embargos_stg
    .withColumn("empresa_id", F.when(F.col("embargo_de") == F.lit("Salario"), F.col("empresa_id_raw")))  # ver supuesto E
    .drop("empresa_id_raw")
    .filter(F.col("cuenta_id").isNotNull())
)
merge_dimension_with_attrs(
    FACT_EMBARGOS, "cuenta_id",
    ["embargo_de_id", "fecha_embargo", "cantidad_mensual_retenida", "empresa_id"],
    embargos_stg,
)


# ---------------------------------------------------------------------------
# 8. Parseo de bitacora de texto ("// dd/mm/aaaa hh:mm:ss usuario / comentario (categoria). // ...")

# ---------------------------------------------------------------------------
LOG_ENTRY_PATTERN = r"^(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2}:\d{2})\s+(\S+)\s*/\s*(.*?)\s*\(([^()]*)\)\.?\s*$"


def parse_gestion_log(df: DataFrame, text_col: str, id_col: str) -> DataFrame:
    """
    Explota un bloque de texto acumulativo en una fila por accion:
    (id_col, fecha_gestion timestamp, usuario, comentario).
    Filas que no calzan con el patron esperado se descartan silenciosamente.
    """
    exploded = (
        df.select(id_col, text_col)
        .filter(F.col(text_col).isNotNull())
        .withColumn("_entrada", F.explode(F.split(F.col(text_col), r"//")))
        .withColumn("_entrada", F.trim(F.col("_entrada")))
        .filter(F.col("_entrada") != "")
    )
    parsed = (
        exploded
        .withColumn("_fecha_str", F.regexp_extract("_entrada", LOG_ENTRY_PATTERN, 1))
        .withColumn("_hora_str", F.regexp_extract("_entrada", LOG_ENTRY_PATTERN, 2))
        .withColumn("usuario", F.regexp_extract("_entrada", LOG_ENTRY_PATTERN, 3))
        .withColumn("comentario", F.trim(F.regexp_extract("_entrada", LOG_ENTRY_PATTERN, 4)))
        .filter(F.col("_fecha_str") != "")
        .withColumn("fecha_gestion", F.to_timestamp(F.concat_ws(" ", "_fecha_str", "_hora_str"), "dd/MM/yyyy HH:mm:ss"))
        .select(F.col(id_col), "fecha_gestion", "usuario", "comentario")
        .filter(F.col("fecha_gestion").isNotNull())
        .dropDuplicates([id_col, "fecha_gestion", "usuario", "comentario"])
    )
    return parsed


def append_detalle_nuevo(parsed_df: DataFrame, fact_table: str):
    """Inserta (append) solo las filas de parsed_df que no existan aun en fact_table,
    comparando por (cuenta_id, fecha_gestion, usuario, comentario)."""
    existentes = spark.table(fact_table).select("cuenta_id", "fecha_gestion", "usuario", "comentario")
    nuevas = parsed_df.join(existentes, on=["cuenta_id", "fecha_gestion", "usuario", "comentario"], how="left_anti")
    nuevas.select("cuenta_id", "fecha_gestion", "usuario", "comentario").write.mode("append").saveAsTable(fact_table)

# ---------------------------------------------------------------------------
# 9. fact_gestion_cuenta + fact_gestion_detalle
# ---------------------------------------------------------------------------
gestion_log_full = parse_gestion_log(cuenta_base, "gestion_diaria", "numero_de_prestamo_vp")
gestion_log_full = attach_id(gestion_log_full, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
gestion_log_full = gestion_log_full.filter(F.col("cuenta_id").isNotNull()).select("cuenta_id", "fecha_gestion", "usuario", "comentario")

ultima_gestion = gestion_log_full.groupBy("cuenta_id").agg(F.max("fecha_gestion").alias("_fecha_ultima_gestion_ts"))

gestion_stg = cuenta_base.select(
    "numero_de_prestamo_vp", "caracterizacion_del_mes", "sub_caracterizacion", "codigos_gestion",
    "fecha_ultima_visita", "gestion_historica", "gestion_diaria",
)
gestion_stg = attach_id(gestion_stg, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
gestion_stg = attach_id(gestion_stg, DIM_CARACT_MENSUAL, ["caracterizacion_mensual"], "caracterizacion_mensual_id", ["caracterizacion_del_mes"], "caracterizacion_mensual_id")
gestion_stg = attach_id(gestion_stg, DIM_SUB_CARACT, ["sub_caracterizacion"], "sub_caracterizacion_id", ["sub_caracterizacion"], "sub_caracterizacion_id")
gestion_stg = attach_id(gestion_stg, DIM_CODIGOS_GESTION, ["codigos_gestion"], "codigos_gestion_id", ["codigos_gestion"], "codigos_gestion_id")
gestion_stg = gestion_stg.join(ultima_gestion, on="cuenta_id", how="left")
gestion_stg = gestion_stg.withColumn("fecha_ultima_gestion", F.to_date(F.col("_fecha_ultima_gestion_ts"))).drop("_fecha_ultima_gestion_ts")
gestion_stg = gestion_stg.filter(F.col("cuenta_id").isNotNull())

merge_dimension_with_attrs(
    FACT_GESTION_CUENTA, "cuenta_id",
    ["caracterizacion_mensual_id", "sub_caracterizacion_id", "codigos_gestion_id",
     "fecha_ultima_visita", "gestion_historica", "gestion_diaria", "fecha_ultima_gestion"],
    gestion_stg,
)

append_detalle_nuevo(gestion_log_full, FACT_GESTION_DETALLE)

# ---------------------------------------------------------------------------
# 10. fact_gestion_judicial_cuenta + fact_gestion_judicial_detalle
# ---------------------------------------------------------------------------
judicial_log_full = parse_gestion_log(cuenta_base, "gestion_judicial_historica", "numero_de_prestamo_vp")
judicial_log_full = attach_id(judicial_log_full, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
judicial_log_full = judicial_log_full.filter(F.col("cuenta_id").isNotNull()).select("cuenta_id", "fecha_gestion", "usuario", "comentario")

judicial_stg = cuenta_base.select(
    "numero_de_prestamo_vp", "tipo_demanda", "estado_demanda", "caracterizacion_judicial",
    "juzgado", "juez", "ciudad_demanda", "expediente", "fecha_solicitud_documento",
    "fecha_recepcion_documentos", "fecha_presentacion_demanda", "cuantia_demandada_lps",
    "cuantia_demandada_usd", "fecha_ultimo_movimiento_judicial", "codigos_gestion",
    "gestion_judicial_historica", "fecha_ultima_revision_expediente", "fecha_requerimiento",
)
judicial_stg = attach_id(judicial_stg, FACT_CUENTA, ["numero_de_prestamo_vp"], "cuenta_id", ["numero_de_prestamo_vp"], "cuenta_id")
judicial_stg = attach_id(judicial_stg, DIM_TIPOS_DEMANDA, ["tipo_demanda"], "tipo_demanda_id", ["tipo_demanda"], "tipo_demanda_id")
judicial_stg = attach_id(judicial_stg, DIM_ESTADOS_DEMANDA, ["estado_demanda"], "estado_demanda_id", ["estado_demanda"], "estado_demanda_id")
judicial_stg = attach_id(judicial_stg, DIM_CARACT_JUDICIAL, ["caracterizacion_judicial"], "caracterizacion_judicial_id", ["caracterizacion_judicial"], "caracterizacion_judicial_id")
judicial_stg = attach_id(judicial_stg, DIM_JUZGADOS, ["juzgado"], "juzgado_id", ["juzgado"], "juzgado_id")
judicial_stg = attach_id(judicial_stg, DIM_JUECES, ["juez"], "juez_id", ["juez"], "juez_id")
judicial_stg = attach_id(judicial_stg, DIM_CIUDADES, ["ciudad"], "ciudad_id", ["ciudad_demanda"], "ciudad_demanda_id")
judicial_stg = attach_id(judicial_stg, DIM_CODIGOS_GESTION, ["codigos_gestion"], "codigos_gestion_id", ["codigos_gestion"], "codigos_gestion_id")
judicial_stg = judicial_stg.filter(F.col("cuenta_id").isNotNull())

merge_dimension_with_attrs(
    FACT_GESTION_JUDICIAL_CUENTA, "cuenta_id",
    ["tipo_demanda_id", "estado_demanda_id", "caracterizacion_judicial_id", "juzgado_id", "juez_id",
     "ciudad_demanda_id", "expediente", "fecha_solicitud_documento", "fecha_recepcion_documentos",
     "fecha_presentacion_demanda", "cuantia_demandada_lps", "cuantia_demandada_usd",
     "fecha_ultimo_movimiento_judicial", "codigos_gestion_id", "gestion_judicial_historica",
     "fecha_ultima_revision_expediente", "fecha_requerimiento"],
    judicial_stg,
)

append_detalle_nuevo(judicial_log_full, FACT_GESTION_JUDICIAL_DETALLE)

print("Gold layer actualizado.")
for t in [
    DIM_RESPONSABLES, DIM_CLIENTES, DIM_CONTACTO_CLIENTES, DIM_SEGMENTACION, DIM_TIPOS_PRODUCTO,
    DIM_PRODUCTOS, DIM_ESTADOS_CARTERA, DIM_TIPOS_EMPRESAS, DIM_EMPRESAS, DIM_BIENES,
    DIM_CARACT_MENSUAL, DIM_SUB_CARACT, DIM_CODIGOS_GESTION, DIM_ESTADOS_PROMESA, DIM_TIPOS_DEMANDA,
    DIM_ESTADOS_DEMANDA, DIM_CARACT_JUDICIAL, DIM_EMBARGOS_DE, DIM_JUZGADOS, DIM_JUECES, DIM_CIUDADES,
    BRIDGE_CIUDADES, DIM_TIPO_ARREGLO, FACT_CUENTA, FACT_PROMESAS, FACT_LABORAL, FACT_GESTION_CUENTA,
    FACT_GESTION_DETALLE, FACT_EMBARGOS, FACT_GESTION_JUDICIAL_CUENTA, FACT_GESTION_JUDICIAL_DETALLE,
]:
    print(f"  {t}: {spark.table(t).count()} filas")
